## UKB Drug response phenotype data extraction

aim: a single table with unique coding for all the participants with available GP records

This part is run performed on the UKB RAP (dnanexus) platform, this needs to be performed once per project

### the main table structure contains all relevant records (hospital, death, GP)


column_title | eid | source | date | code | system | info |
----- | ----- | ----- | ------ |----- | ----- | ------ |
contents | participant info | GP / PRESC / HOSP | date of record | assigined code | coding system ICD10 ICD9 etc | drug dose other free text |

The sql query in this notebook performs the following:

1. collect all the participant ids with primary health records 
2. collect diagnosis codes with date for all above participants from hospital records 
3. collect diagnosis codes with date for all above participants from GP records 
4. collect prescription codes with date and dose for all above participants 
5. collect diagnosis codes with date for all above participants from death records

this results in a database where each assigned code is in its native form, without any recoding - this provides the fullest information. this exists as an sql query and a hail matrixtable.

In [ ]:
import pyspark
import dxpy
import dxdata
sc = pyspark.SparkContext()

spark = pyspark.sql.SparkSession(sc)

Get database info:

In [ ]:
dispensed_database_name = dxpy.find_one_data_object(
    classname="database", name="app*", folder="/", name_mode="glob", describe=True
)["describe"]["name"]

In [ ]:
spark.sql("USE " + dispensed_database_name)

#### STEP1: Get ID of all participants with GP data

tables we will use: 

- gp_registrations (to get dates when people registered)
- hesin (hospital diag dates)
- hesin_diag (hospital diagnoses, icd9 or idc10)
- death (death dates)
- death_cause (death cause icd10)
- gp_clinical (diagnoses and procedures from GP records)
- gp_scripts (GP prescriptions)


In [ ]:
test_table = spark.sql("""
            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.dmd_code AS code,
            presc.issue_date AS date,
            'dmd' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.dmd_code IS NOT NULL
            
            UNION
            
            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.bnf_code AS code,
            presc.issue_date AS date,
            'bnf' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.bnf_code IS NOT NULL
            
            UNION

            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.read_2 AS code,
            presc.issue_date AS date,
            'read_2' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.read_2 IS NOT NULL
            
            UNION

            SELECT 
            gp.eid AS eid,
            'HOSP' as source,
            hesin_diag.diag_icd10 AS code,
            COALESCE(hesin.epistart, hesin.admidate, hesin.epiend, hesin.disdate) AS date,
            'ICD10' AS system,
            'none' AS info

            FROM gp_registrations AS gp
            
            JOIN hesin_diag ON gp.eid = hesin_diag.eid
            JOIN hesin ON hesin_diag.dnx_hesin_id = hesin.dnx_hesin_id
            AND hesin_diag.diag_icd10 IS NOT NULL
            
            UNION
            
            SELECT gp.eid AS eid,
            'HOSP' as source,
            hesin_diag.diag_icd9 AS code,
            COALESCE(hesin.epistart, hesin.admidate, hesin.epiend, hesin.disdate) AS date,
            'ICD9' AS system,
            'none' AS info
            FROM gp_registrations AS gp
            JOIN hesin_diag ON gp.eid = hesin_diag.eid
            JOIN hesin ON hesin_diag.dnx_hesin_id = hesin.dnx_hesin_id
            AND hesin_diag.diag_icd9 IS NOT NULL
            
            UNION
            
            SELECT gp.eid AS eid,
            'MORT' as source,
            death_cause.cause_icd10 AS code,
            death.date_of_death AS date,
            'ICD10' AS system,
            'none' AS info
            FROM gp_registrations AS gp
            JOIN death ON gp.eid = death.eid
            JOIN death_cause ON death.dnx_death_id = death_cause.dnx_death_id
            AND death_cause.cause_icd10 IS NOT NULL
            
            UNION
            
            SELECT gp_c.eid AS eid,
            'GP' as source,
            gp_c.read_2 AS code,
            gp_c.event_dt AS date,
            'read_2' AS system,
            CONCAT (COALESCE(gp_c.value1, ''), '_', COALESCE(gp_c.value2, ''), '_', COALESCE(gp_c.value3, '')) as info
            FROM gp_clinical AS gp_c
            WHERE gp_c.read_2 IS NOT NULL
            
            UNION
            
            SELECT gp_c.eid AS eid,
            'GP' as source,
            gp_c.read_3 AS code,
            gp_c.event_dt AS date,
            'read_3' AS system,
            CONCAT (COALESCE(gp_c.value1, ''), '_', COALESCE(gp_c.value2, ''), '_', COALESCE(gp_c.value3, '')) as info
            FROM gp_clinical AS gp_c
            WHERE gp_c.read_3 IS NOT NULL
            
            LIMIT 10000""")

In [ ]:
full_table = spark.sql("""
            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.dmd_code AS code,
            presc.issue_date AS date,
            'dmd' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.dmd_code IS NOT NULL
            
            UNION
            
            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.bnf_code AS code,
            presc.issue_date AS date,
            'bnf' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.bnf_code IS NOT NULL
            
            UNION

            SELECT presc.eid AS eid,
            'PRESC' as source,
            presc.read_2 AS code,
            presc.issue_date AS date,
            'read_2' AS system,
            CONCAT (COALESCE(presc.drug_name, ''), '_', COALESCE(presc.quantity, '')) as info
            FROM gp_scripts AS presc
            WHERE presc.read_2 IS NOT NULL
            
            UNION

            SELECT 
            gp.eid AS eid,
            'HOSP' as source,
            hesin_diag.diag_icd10 AS code,
            COALESCE(hesin.epistart, hesin.admidate, hesin.epiend, hesin.disdate) AS date,
            'ICD10' AS system,
            'none' AS info

            FROM gp_registrations AS gp
            
            JOIN hesin_diag ON gp.eid = hesin_diag.eid
            JOIN hesin ON hesin_diag.dnx_hesin_id = hesin.dnx_hesin_id
            AND hesin_diag.diag_icd10 IS NOT NULL
            
            UNION
            
            SELECT gp.eid AS eid,
            'HOSP' as source,
            hesin_diag.diag_icd9 AS code,
            COALESCE(hesin.epistart, hesin.admidate, hesin.epiend, hesin.disdate) AS date,
            'ICD9' AS system,
            'none' AS info
            FROM gp_registrations AS gp
            JOIN hesin_diag ON gp.eid = hesin_diag.eid
            JOIN hesin ON hesin_diag.dnx_hesin_id = hesin.dnx_hesin_id
            AND hesin_diag.diag_icd9 IS NOT NULL
            
            UNION
            
            SELECT gp.eid AS eid,
            'MORT' as source,
            death_cause.cause_icd10 AS code,
            death.date_of_death AS date,
            'ICD10' AS system,
            'none' AS info
            FROM gp_registrations AS gp
            JOIN death ON gp.eid = death.eid
            JOIN death_cause ON death.dnx_death_id = death_cause.dnx_death_id
            AND death_cause.cause_icd10 IS NOT NULL
            
            UNION
            
            SELECT gp_c.eid AS eid,
            'GP' as source,
            gp_c.read_2 AS code,
            gp_c.event_dt AS date,
            'read_2' AS system,
            CONCAT (COALESCE(gp_c.value1, ''), '_', COALESCE(gp_c.value2, ''), '_', COALESCE(gp_c.value3, '')) as info
            FROM gp_clinical AS gp_c
            WHERE gp_c.read_2 IS NOT NULL
            
            UNION
            
            SELECT gp_c.eid AS eid,
            'GP' as source,
            gp_c.read_3 AS code,
            gp_c.event_dt AS date,
            'read_3' AS system,
            CONCAT (COALESCE(gp_c.value1, ''), '_', COALESCE(gp_c.value2, ''), '_', COALESCE(gp_c.value3, '')) as info
            FROM gp_clinical AS gp_c
            WHERE gp_c.read_3 IS NOT NULL
            """)

In [ ]:
import hail as hl
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from pyspark.sql.functions import col

test_table = (
    test_table
    .withColumn('date', col('date').cast('string'))
)

full_table = (
    full_table
    .withColumn('date', col('date').cast('string'))
)

In [ ]:
test_table = hl.Table.from_spark(test_table)

In [ ]:
full_table = hl.Table.from_spark(full_table)

In [ ]:
#create a database and save a test table and the full table:

db_name = "arb_db" #this is the baseline database for this project
test_tb_name = "base_subset_10k.ht"
full_tb_name = "base.ht"

stmt = f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION 'dnax://'"
print(stmt)

spark.sql(stmt).show()

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url1 = f"dnax://{db_uri}/{test_tb_name}"
url2 = f"dnax://{db_uri}/{full_tb_name}"

In [ ]:
test_table.write(url1, overwrite = True)

In [ ]:
full_table.write(url2, overwrite = True)